# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready GeoParquet row per parcel with dated feature columns.

In [1]:
from pathlib import Path
import geopandas as gpd

from satellites.data_preparation.parcel_stats import estimate_utm_epsg_from_parcels
from satellites.data_preparation.parcel_stats.multiuser import (
    load_openeo_users_from_env,
    run_parallel_extractions,
    merge_and_save_results,
)
from satellites.data_preparation.parcel_stats.job_manager import (
    compute_tile_buffer_metres,
)
from satellites.shared.io import read_data, write_data
import satellites.shared.geometry as geom_l
import satellites.shared.spatial_statistics as stat_l

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`acquisition pixel indices`
`monthly pixel median`
`monthly per-variable cube IQR outlier removal`
`local null filling`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [6]:
# ============================================================================
# STEP 1: Read data and re-project
# ============================================================================
# Read the data and re-project to the desired EPSG

WORK_PATH = Path("C:/work_dir")
RUN_IDENTIFIER = "neuro_2024-2025"


PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "sample_parcels.csv"
gdf_parcels = read_data(str(PARCELS_PATH))
print("Initial parcels data:")
display(gdf_parcels.head())

gdf_parcels = gpd.GeoDataFrame(gdf_parcels, geometry=gpd.GeoSeries.from_wkt(gdf_parcels["geom_wkt"]), crs="EPSG:2100")
print(f"Number of parcels: {len(gdf_parcels)}")

gdf_parcels = gdf_parcels.drop_duplicates()
print(f"Number of parcels after dropping duplicates: {len(gdf_parcels)}")
gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(f"Number of parcels after ensuring valid geometries and converting multipolygons to polygons: {len(gdf_parcels)}")
gdf_parcels = gdf_parcels.drop(columns=["geom_wkt"])
write_data(gdf_parcels, str(PARCELS_PATH).replace(".csv", ".gpkg"))

# PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "sample_parcels.gpkg"
# gdf_parcels = read_data(str(PARCELS_PATH))
display(gdf_parcels)

Reading file: C:/work_dir/neuro_2024-2025/sample_parcels.csv
Initial parcels data:


,parcel_id,crop_category,geom_wkt
0,552,ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ,"MULTIPOLYGON(((276662.481 4436671.199,276678.7..."
1,552,ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ,"MULTIPOLYGON(((276662.481 4436671.199,276678.7..."
2,552,ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ,"MULTIPOLYGON(((276662.481 4436671.199,276678.7..."
3,552,ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ,"MULTIPOLYGON(((276662.481 4436671.199,276678.7..."
4,552,ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ,"MULTIPOLYGON(((276662.481 4436671.199,276678.7..."


Number of parcels: 5662
Number of parcels after dropping duplicates: 1177
Number of parcels after ensuring valid geometries and converting multipolygons to polygons: 1177
Removing existing file: C:/work_dir/neuro_2024-2025/sample_parcels.gpkg
Writing file: C:/work_dir/neuro_2024-2025/sample_parcels.gpkg


,parcel_id,crop_category,geometry
0,552,ΕΝΕΡΓΕΙΑΚΕΣ ΚΑΛΛΙΕΡΓΕΙΕΣ,"POLYGON ((276662.481 4436671.199, 276678.713 4..."
1,553,jjjj,"POLYGON ((278512.622 4438032.513, 278569.445 4..."
2,554,ΚΤΗΝΟΤΡΟΦΙΚΑ ΦΥΤΑ,"POLYGON ((278217.407 4437980.985, 278216.142 4..."
3,555,ΚΤΗΝΟΤΡΟΦΙΚΑ ΦΥΤΑ,"POLYGON ((274769.372 4440021.269, 274765.086 4..."
4,556,ΚΤΗΝΟΤΡΟΦΙΚΑ ΦΥΤΑ,"POLYGON ((272117.609 4439208.353, 272117.343 4..."
...,...,...,...
1172,25568,ΑΡΑΒΟΣΙΤΟΣ,"POLYGON ((279312.404 4518258.109, 279282.821 4..."
1173,25601,ΕΛΑΙΩΝΕΣ,"POLYGON ((617782.701 4067252.962, 617772.565 4..."
1174,25602,ΠΟΡΤΟΚΑΛΙΑ ΠΡΟΣ ΧΥΜΟΠΟΙΗΣΗ,"POLYGON ((625014.344 3873755.626, 625003.543 3..."
1175,25651,ΒΑΜΒΑΚΙ,"POLYGON ((700907.829 4610302.442, 700796.329 4..."


In [ ]:
print(gdf_parcels.crs)
WORKING_EPSG = estimate_utm_epsg_from_parcels(gdf_parcels)
WORKING_EPSG

In [ ]:
# ============================================================================
# Define the class column and parcel ID column names.
# These are used in the extraction process to identify the class of each parcel
# and its unique identifier.
# ============================================================================
class_column = "crop_category"
parcel_id_column = "parcel_id"

In [ ]:
# Reuse saved cluster labels; generate them only when the input has none.
CLUSTER_FIELD = "cluster"
if CLUSTER_FIELD in gdf_parcels.columns:
    gdf_grouped = gdf_parcels.copy()
else:
    gdf_grouped = stat_l.spatial_clustering_using_buffer(
        gdf_parcels.to_crs(epsg=WORKING_EPSG),
        parcel_id_column,
        min_cluster_distance_in_m=50_000,
        field_cluster=CLUSTER_FIELD,
        plot_results=False,
    )

# groupby would silently omit parcels with missing labels.
if gdf_grouped[CLUSTER_FIELD].isna().any():
    raise ValueError("Every parcel must have a cluster label before partitioning.")


## Multi-user extraction planning

Create one spatial partition per cluster. Existing user queues distribute these
partitions across accounts. With `fit_tiles_to_parcels=True`, a cube covers the
group's complete bounding box plus `TILE_BUFFER_METRES` on every side:

- width = east - west + 2 * buffer
- height = north - south + 2 * buffer

Only if either dimension exceeds `TILE_SIZE_METRES` does the planner split the
group spatially, then fit and check each subgroup in the same way. Parcels stay
whole and belong to exactly one job. A single parcel that exceeds the limit with
its buffer raises an error before submission.

Small clusters may use fewer remote slots, and accounts can be idle when there
are fewer clusters than users.


In [ ]:
# ============================================================================
# STEP 2: Load accounts for cluster partitions
# ============================================================================
# Cluster partitions are assigned to the existing per-user queues.

OPENEO_JOBS_PER_USER = 2  # This is default by openEO
users_list = load_openeo_users_from_env()

print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")

In [ ]:
# Keep each cluster together; the existing tile planner handles its size limit.
spatial_parts = [
    part.copy()
    for _, part in gdf_grouped.groupby(CLUSTER_FIELD, sort=True, observed=True)
]
assert sum(len(part) for part in spatial_parts) == len(gdf_grouped)

print(f"Created {len(spatial_parts)} cluster partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {part[CLUSTER_FIELD].iloc[0]} - {len(part)} parcels")


In [ ]:
# Maximum requested cube width AND height, including the buffer.
# Each group gets its own fitted extent; only oversized groups are split.
TILE_SIZE_METRES = 20_000
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_grouped, WORKING_EPSG)


In [ ]:
print(f"Maximum buffered cube width/height: {TILE_SIZE_METRES:,} m")
print(f"Buffer on each side: {TILE_BUFFER_METRES:,} m")


In [ ]:
# ============================================================================
# Confirm the planned global remote concurrency.
# ============================================================================
TOTAL_REMOTE_JOB_CAPACITY = len(users_list) * OPENEO_JOBS_PER_USER
print(
    f"Remote capacity: {len(users_list)} users x {OPENEO_JOBS_PER_USER} jobs "
    f"= {TOTAL_REMOTE_JOB_CAPACITY} simultaneous jobs"
)


In [ ]:
# ============================================================================
# STEP 3: Create extraction configuration dictionary
# ============================================================================
# Single dictionary containing all parameters for the extraction process,
# to be passed to each parallel worker.

START_DATE = "2025-03-01"
END_DATE = "2025-12-31"
OUTPUT_DIR = WORK_PATH / RUN_IDENTIFIER / "1_eo_parcel_stats_fill"

SENTINEL2_BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
SENTINEL2_INDICES = ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"]

SENTINEL1_BANDS = [] # ["VV", "VH"]
SENTINEL1_INDICES = [] #  ["R", "RVI"]
SENTINEL1_ORBIT_DIRECTION = "ASCENDING"  # ASCENDING, DESCENDING, or BOTH

# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    "range",
]

INTERPOLATION_METHOD = "nearest"  # "idw"
INTERPOLATION_MAX_DISTANCE_IN_METERS = 500

# Temporal period and reducer for the temporal aggregation of the data. "1M" means monthly aggregation, and "none" means no temporal reduction (keep all observations).
TEMPORAL_PERIOD = "1M"
TEMPORAL_REDUCER = "none"

# IQR outliers become null, then nulls are filled.
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
IQR_MIN_VALID_PIXELS = 20
REMOVE_OUTLIERS = False

FILL_NULLS = True
TEMPORAL_FILL_MODE = "bidirectional" # or past_only
MINIMUM_PARCEL_PIXELS = 3
MINIMUM_OBSERVED_FRACTION_FOR_FILL = 0.20

# Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
LOCAL_BATCH_WORKERS = 2

extraction_config = {
    # Output paths
    "output_dir": OUTPUT_DIR,
    "run_identifier": RUN_IDENTIFIER,

    # Parcel configuration
    "parcel_id_field": parcel_id_column,

    # Temporal bounds
    "start_date": START_DATE,
    "end_date": END_DATE,

    # Spatial configuration
    "working_epsg": WORKING_EPSG,

    # Sentinel-2 configuration
    "sentinel2_bands": SENTINEL2_BANDS,
    "sentinel2_indices": SENTINEL2_INDICES,

    # Sentinel-1 configuration
    "sentinel1_bands": SENTINEL1_BANDS,
    "sentinel1_indices": SENTINEL1_INDICES,
    "sentinel1_orbit_direction": SENTINEL1_ORBIT_DIRECTION,

    # Statistics to calculate
    "spatial_statistics": SPATIAL_STATISTICS,

    # Fit cluster bounds plus buffer; split only above the maximum size.
    "fit_tiles_to_parcels": True,
    "tile_size_metres": TILE_SIZE_METRES,
    "tile_buffer_metres": TILE_BUFFER_METRES,

    # Data cleaning (IQR outlier removal)
    "remove_outliers": REMOVE_OUTLIERS,
    "iqr_quantiles": IQR_QUANTILES,
    "iqr_multiplier": IQR_MULTIPLIER,
    "iqr_min_valid_pixels": IQR_MIN_VALID_PIXELS,

    # Null filling
    "fill_nulls": FILL_NULLS,
    "temporal_fill_mode": TEMPORAL_FILL_MODE,
    "minimum_parcel_pixels": MINIMUM_PARCEL_PIXELS,
    "minimum_observed_fraction_for_fill": MINIMUM_OBSERVED_FRACTION_FOR_FILL,
    "temporal_period": TEMPORAL_PERIOD,
    "temporal_reducer": TEMPORAL_REDUCER,

    # Interpolation
    "interpolation_method": INTERPOLATION_METHOD,
    "interpolation_max_distance_in_meters": INTERPOLATION_MAX_DISTANCE_IN_METERS,

    # Parallel processing
    "openeo_parallel_jobs": OPENEO_JOBS_PER_USER,
    "job_poll_seconds": 30,
    "batch_workers": LOCAL_BATCH_WORKERS,

    # Other options
    "spatial_fill_window_sizes": (3, 5),
}

In [ ]:
# from satellites.data_preparation.parcel_stats.multiuser import run_partition_extractor

# run_partition_extractor(
#     part_idx=14,
#     gdf_part=spatial_parts[13],
#     users_list=users_list,
#     total_partitions=len(spatial_parts),
#     config={**extraction_config, "batch_workers": 1},
# )

In [ ]:
# ============================================================================
# STEP 4: Run all partitions in parallel
# ============================================================================
# Users run in parallel; each user's assigned partitions run sequentially.
# Each active partition manager can run two openEO jobs simultaneously.
run_parallel_extractions(
    spatial_parts=spatial_parts,
    users_list=users_list,
    config=extraction_config,
)

# Each partition result is saved below OUTPUT_DIR/partition_<n>.
print("\nExtraction complete!")

In [ ]:
# ============================================================================
# STEP 4: Merge results and save to files
# ============================================================================
# Combines all partition results and removes duplicates
# Saves one file:
#   1. satellite_parcel_time_stats.geoparquet (full results with geometry)

merged_gdf, geoparquet_path = merge_and_save_results(
    output_dir=OUTPUT_DIR,
    parcel_id_column=parcel_id_column,
    working_epsg=WORKING_EPSG
)

In [ ]:
# ============================================================================
# STEP 5: Inspect results
# ============================================================================

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")

print("\nMerged GeoDataFrame:")
print(f"  Rows: {len(merged_gdf)}")
print(f"  Columns: {len(merged_gdf.columns)}")
print(f"  Has geometry: {merged_gdf.geometry.name in merged_gdf.columns}")

print("\nOutput files:")
print(f"  GeoParquet: {geoparquet_path}")
print(f"    Size: {geoparquet_path.stat().st_size / 1e6:.1f} MB")

print("\nFirst 5 rows:")
display(merged_gdf.head())